In [1]:
# STEP 1: Load the data
import pandas as pd

df = pd.read_csv("/content/customer_churn_dataset-testing-master.csv")   # <- change path if yours differs
print("Rows, columns:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Churn rate: {:.1%}".format(df["Churn"].mean()))
df.head()


Rows, columns: (64374, 12)
Missing values: 0
Churn rate: 47.4%


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [2]:
# STEP 2: Prepare the data
# - CustomerID is just a label, so we drop it
# - Text columns (Gender, Subscription Type, Contract Length) become 0/1 columns
X = pd.get_dummies(df.drop(columns=["CustomerID", "Churn"]), drop_first=True)
y = df["Churn"]          # 1 = churned, 0 = stayed

# Hold back 20% of customers to test the model on data it has never seen
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training customers:", len(X_train), "| Testing customers:", len(X_test))



Training customers: 51499 | Testing customers: 12875


In [3]:
# STEP 3: Train the model
# max_depth=5 keeps the tree small enough to read AND stops it memorising the data
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)


DecisionTreeClassifier(max_depth=5, random_state=42)

In [4]:
# STEP 4: How good is it?  (tested on the unseen 20%)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pred = model.predict(X_test)
print("Accuracy: {:.1%}\n".format(accuracy_score(y_test, pred)))
print(classification_report(y_test, pred, target_names=["Stayed", "Churned"]))

cm = confusion_matrix(y_test, pred)
print("Confusion matrix (rows = actual, columns = predicted)")
print(pd.DataFrame(cm, index=["Actually stayed", "Actually churned"],
                       columns=["Pred stayed", "Pred churned"]))

Accuracy: 95.4%

              precision    recall  f1-score   support

      Stayed       0.98      0.93      0.96      6776
     Churned       0.93      0.98      0.95      6099

    accuracy                           0.95     12875
   macro avg       0.95      0.95      0.95     12875
weighted avg       0.95      0.95      0.95     12875

Confusion matrix (rows = actual, columns = predicted)
                  Pred stayed  Pred churned
Actually stayed          6334           442
Actually churned          152          5947


In [5]:
# STEP 5: EXPLAIN the model - what drives churn?
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Most important factors (higher = matters more):\n")
print(importance.head(6).round(3).to_string())

# The model's actual rules, in plain text
from sklearn.tree import export_text
print("\nTop of the decision tree:\n")
print(export_text(model, feature_names=list(X.columns), max_depth=2))


Most important factors (higher = matters more):

Payment Delay      0.485
Support Calls      0.144
Gender_Male        0.115
Tenure             0.100
Usage Frequency    0.090
Age                0.044

Top of the decision tree:

|--- Payment Delay <= 15.50
|   |--- Usage Frequency <= 5.50
|   |   |--- Age <= 50.50
|   |   |   |--- truncated branch of depth 3
|   |   |--- Age >  50.50
|   |   |   |--- truncated branch of depth 3
|   |--- Usage Frequency >  5.50
|   |   |--- class: 0
|--- Payment Delay >  15.50
|   |--- Support Calls <= 4.50
|   |   |--- Gender_Male <= 0.50
|   |   |   |--- truncated branch of depth 3
|   |   |--- Gender_Male >  0.50
|   |   |   |--- truncated branch of depth 3
|   |--- Support Calls >  4.50
|   |   |--- Tenure <= 23.50
|   |   |   |--- truncated branch of depth 3
|   |   |--- Tenure >  23.50
|   |   |   |--- class: 1



In [6]:
# STEP 6: Predict churn risk for ANY new customer
def predict_churn(age, gender, tenure, usage_frequency, support_calls,
                  payment_delay, subscription_type, contract_length,
                  total_spend, last_interaction):

    # 1) Put the inputs in a one-row table with the SAME column names as the training data
    customer = pd.DataFrame([{
        "Age": age, "Gender": gender, "Tenure": tenure,
        "Usage Frequency": usage_frequency, "Support Calls": support_calls,
        "Payment Delay": payment_delay, "Subscription Type": subscription_type,
        "Contract Length": contract_length, "Total Spend": total_spend,
        "Last Interaction": last_interaction,
    }])

    # 2) Encode text the same way as training; reindex guarantees identical columns
    customer = pd.get_dummies(customer).reindex(columns=X.columns, fill_value=0)

    # 3) Probability of churn -> risk level
    prob = model.predict_proba(customer)[0][1]
    level = "LOW" if prob < 0.30 else "MEDIUM" if prob < 0.70 else "HIGH"

    # 4) Explain: which rules did THIS customer pass through?
    tree = model.tree_
    reasons = []
    for node in model.decision_path(customer).indices[:-1]:      # skip the final leaf
        feature = X.columns[tree.feature[node]]
        value = customer.iloc[0][feature]
        if value <= tree.threshold[node]:
            reasons.append(f"{feature} = {value:g}  (<= {tree.threshold[node]:.1f})")
        else:
            reasons.append(f"{feature} = {value:g}  (> {tree.threshold[node]:.1f})")

    print(f"Churn probability : {prob:.0%}")
    print(f"Risk level        : {level}")
    print("Why (rules followed):")
    for r in reasons:
        print("   -", r)
    print()
    return prob


In [7]:
# STEP 7: Try it - change these numbers to test your own customers
print("=== Customer A: unhappy-looking profile ===")
predict_churn(age=30, gender="Female", tenure=5, usage_frequency=4, support_calls=9,
              payment_delay=25, subscription_type="Basic", contract_length="Monthly",
              total_spend=300, last_interaction=25)

print("=== Customer B: loyal-looking profile ===")
predict_churn(age=45, gender="Male", tenure=48, usage_frequency=25, support_calls=1,
              payment_delay=2, subscription_type="Premium", contract_length="Annual",
              total_spend=850, last_interaction=3)

print("=== Customer C: borderline profile ===")
predict_churn(age=45, gender="Female", tenure=21, usage_frequency=26, support_calls=4,
              payment_delay=22, subscription_type="Premium", contract_length="Quarterly",
              total_spend=628, last_interaction=2)

=== Customer A: unhappy-looking profile ===
Churn probability : 85%
Risk level        : HIGH
Why (rules followed):
   - Payment Delay = 25  (> 15.5)
   - Support Calls = 9  (> 4.5)
   - Tenure = 5  (<= 23.5)
   - Gender_Male = 0  (<= 0.5)
   - Payment Delay = 25  (> 20.5)

=== Customer B: loyal-looking profile ===
Churn probability : 0%
Risk level        : LOW
Why (rules followed):
   - Payment Delay = 2  (<= 15.5)
   - Usage Frequency = 25  (> 5.5)

=== Customer C: borderline profile ===
Churn probability : 57%
Risk level        : MEDIUM
Why (rules followed):
   - Payment Delay = 22  (> 15.5)
   - Support Calls = 4  (<= 4.5)
   - Gender_Male = 0  (<= 0.5)
   - Payment Delay = 22  (> 20.5)
   - Total Spend = 628  (> 500.5)



np.float64(0.5673884224480777)

In [9]:
# STEP 8: Interactive form - move sliders, pick from dropdowns, click Predict
import ipywidgets as widgets
from IPython.display import display

style  = {"description_width": "170px"}
layout = widgets.Layout(width="480px")

age           = widgets.IntSlider(value=30, min=18, max=65, description="Age", style=style, layout=layout)
gender        = widgets.Dropdown(options=["Female", "Male"], description="Gender", style=style, layout=layout)
tenure        = widgets.IntSlider(value=12, min=1, max=60, description="Tenure (months)", style=style, layout=layout)
usage         = widgets.IntSlider(value=15, min=1, max=30, description="Usage frequency", style=style, layout=layout)
support_calls = widgets.IntSlider(value=3, min=0, max=10, description="Support calls", style=style, layout=layout)
payment_delay = widgets.IntSlider(value=10, min=0, max=30, description="Payment delay (days)", style=style, layout=layout)
subscription  = widgets.Dropdown(options=["Basic", "Standard", "Premium"], description="Subscription", style=style, layout=layout)
contract      = widgets.Dropdown(options=["Monthly", "Quarterly", "Annual"], description="Contract length", style=style, layout=layout)
total_spend   = widgets.BoundedIntText(value=500, min=100, max=1000, description="Total spend", style=style, layout=layout)
last_contact  = widgets.IntSlider(value=15, min=1, max=30, description="Days since last contact", style=style, layout=layout)

button = widgets.Button(description="Predict", button_style="primary", layout=widgets.Layout(width="200px", height="40px"))
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        output.clear_output()                       # remove the previous result
        predict_churn(
            age=age.value, gender=gender.value, tenure=tenure.value,
            usage_frequency=usage.value, support_calls=support_calls.value,
            payment_delay=payment_delay.value, subscription_type=subscription.value,
            contract_length=contract.value, total_spend=total_spend.value,
            last_interaction=last_contact.value,
        )

button.on_click(on_predict_clicked)

display(widgets.VBox([age, gender, tenure, usage, support_calls, payment_delay,
                      subscription, contract, total_spend, last_contact, button, output]))